# 01. K6 Lite: вся система и training map

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf) · [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** восстановить состав checkpoint и посчитать параметры каждого потока без выделения памяти под 3B весов

**Перед началом:** вводный probability notebook; K5 00. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/00_start_and_training_map.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.1–3.2, 4](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)
- [`core/components/piflow_dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/piflow_dit.py)
- [`runtime/sampler.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/runtime/sampler.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '01_system_and_training_map'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Модель из 3B параметров — это ещё не весь pipeline

Сначала текст превращается в Qwen token states и CLIP pooled state. Два **разных обучаемых projection/refiner**
внутри DiT готовят их для видео и аудио. Gaussian noise живёт в двух разных latent spaces.
CrossDiT обновляет оба состояния; затем Hunyuan VAE декодирует видео, MMAudio VAE — mel,
а отдельный vocoder — waveform. Mux только объединяет результаты.

Это совместная модель распределения видео и аудио при данном тексте. Отдельный audio-forward
не содержит межмодального обмена. Он полезен для контроля кода, но не воспроизводит полную AV систему.

### Что именно обучается

В нашем полном FM-опыте обучаем DiT; внешние Qwen/CLIP/кодеки frozen. Это явный контракт опыта.
Исторические стадии в статье отличаются: separate → joint pretraining → SFT → RL → distillation.
Особенно сравните §4.3: в SFT stage 1 заморожены audio и fusion, в stage 2 DiT-модули размораживаются.
Не переносите слово «frozen» с одного этапа на всю историю модели. Inference вообще не имеет optimizer.


In [ ]:
index = json.loads((ROOT/'reference/hf/Kandinsky-6.0-Lite-distill-5s-Diffusers/model_index.json').read_text())
for name in ['transformer','text_encoder','tokenizer','text_encoder_2','tokenizer_2','vae','audio_vae','vocoder','scheduler']:
    print(f'{name:16s}', index[name])
cfg=config(); print('video/audio widths:',cfg['model_dim'],cfg['model_dim_a'])
assert cfg['out_visual_dim']==160 and cfg['out_audio_dim']==400
assert cfg['out_visual_dim']//cfg['in_visual_dim']==cfg['out_audio_dim']//cfg['in_audio_dim']==10


## 2. Считать параметры можно без загрузки весов

`meta` хранит формы без значений. Конструктор создаёт правильную структуру, а `numel()` считает элементы.
Из checkpoint config убираем `scale_factor`: это параметр RoPE при вызове, не аргумент конструктора.
Используем **обычный native DiT с уже расширенными выходами 160/400**. Если передать эти выходы
PiFlow-wrapper, который сам умножает на 10, получится ошибочное двойное расширение.

Всё, что начинается с `audio_` или принадлежит `audio_dec_block`, относим к аудиотракту;
`va_`/`av_` — к fusion; остальное — к видео. Это прозрачная группировка state_dict,
а не заявление о независимых файлах весов. В опубликованном transformer они объединены.
Обзорные 2B/0.6B/0.4B из статьи — округлённые архитектурные оценки. Наш numel включает
все instantiated parameters и явно относит cross-modal modulation к fusion; эти группировки не тождественны.


In [ ]:
kwargs={k:v for k,v in cfg.items() if k!='scale_factor'}
kwargs['attention_engine']='sdpa'
with torch.device('meta'):
    full=native().DiffusionTransformer3D(**kwargs)
counts={'video':0,'audio':0,'fusion':0}
for name,p in full.named_parameters():
    group=('fusion' if any(s in name for s in ['.va_','.av_']) else
           'audio' if name.startswith('audio_') or '.audio_dec_block.' in name else 'video')
    counts[group]+=p.numel()
print('Exact parameter count:',counts,'total=',sum(counts.values()))
metrics['parameters']=counts;metrics['total_parameters']=sum(counts.values())
metrics['bf16_transformer_only_GiB']=sum(counts.values())*2/2**30
print('BF16 weights only GiB:',metrics['bf16_transformer_only_GiB'])
fig,ax=plt.subplots(figsize=(7,3));ax.bar(counts.keys(),[n/1e9 for n in counts.values()])
ax.set_ylabel('B parameters');ax.set_title('Lite Distill: counted from native constructor')
fig.tight_layout();fig.savefig(figure_path(LAB,'parameter_groups'));plt.show()
del full


## 3. Связь с ресурсами

BF16-байты параметров — только нижняя оценка хранения DiT. Активации, attention, Qwen,
VAE, allocator и optimizer добавляют память. Unified Memory — общий бюджет GPU/CPU,
а не сумма независимых VRAM и RAM. Adam training может быть существенно тяжелее inference.
SR и prompt beautifier — дополнительные ветви, не обязательные для минимального Lite baseline.


## Самостоятельное исследование

1. Сравните plain Lite и Distill configs: какие формы головы изменились, какие backbone widths остались?
2. Составьте три optimizer maps: полный FM, SFT stage 1, наш frozen-FiLM эксперимент.
3. Добавьте память gradients и двух Adam moments в FP32; подпишите, что активации ещё не учтены.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы можете пройти путь от prompt до MP4 и назвать для каждого узла источник весов,
вход/выход, trainable/frozen статус в выбранном опыте. Число «3B» не является оценкой размера всего pipeline.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[02. Текст: tokenizer, Qwen, CLIP и два refiner →](02_text_conditioning.ipynb)
